# Predicción del Precio Futuro de TSLA
## Proyecto de Machine Learning - Regresión Lineal

**Instituto Tecnológico de Las Américas (ITLA)**  
**Curso:** Inteligencia Artificial  
**Profesor:** Ramón E. Alvarez S.  
**Fecha:** Julio 2026

---

### Pipeline del Proyecto
1. Recolección de Datos (API Alpaca)
2. Preprocesamiento y Ingeniería de Características
3. Análisis Exploratorio de Datos (EDA)
4. Entrenamiento de 8 Modelos de Regresión
5. Evaluación y Comparación
6. Predicción con Nuevos Datos

---
## PARTE 1: RECOLECCIÓN Y PREPROCESAMIENTO DE DATOS
### Responsable: Persona 1

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("ALPACA_API_KEY")
secret_key = os.getenv("ALPACA_SECRET_KEY")

print("Keys cargadas:", api_key is not None, secret_key is not None)

In [ ]:
!pip install alpaca-py

In [ ]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime

In [ ]:
client = StockHistoricalDataClient(api_key, secret_key)

In [ ]:
request_params = StockBarsRequest(
    symbol_or_symbols="TSLA",
    timeframe=TimeFrame.Day,
    start=datetime(2020, 1, 1),
    end=datetime(2026, 7, 21)
)

bars = client.get_stock_bars(request_params)
df = bars.df

print(df.shape)
df.head()

In [ ]:
df.to_csv("../data/raw/tsla_raw.csv")
print("Guardado correctamente")

In [ ]:
df = df.sort_index()

df['retorno_diario'] = df['close'].pct_change()
df['sma_5'] = df['close'].rolling(window=5).mean()
df['sma_20'] = df['close'].rolling(window=20).mean()
df['volatilidad_5'] = df['close'].rolling(window=5).std()
df['volumen_promedio_5'] = df['volume'].rolling(window=5).mean()
df['precio_anterior'] = df['close'].shift(1)
df['precio_futuro'] = df['close'].shift(-1)

print(df.shape)
df.head(10)

In [ ]:
print(df.isnull().sum())

df_limpio = df.dropna()

print("Antes:", df.shape)
print("Después de limpiar:", df_limpio.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['open', 'high', 'low', 'close', 'volume', 'retorno_diario',
            'sma_5', 'sma_20', 'volatilidad_5', 'volumen_promedio_5', 'precio_anterior']

X = df_limpio[features]
y = df_limpio['precio_futuro']

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

print("Forma de X escalado:", X_escalado.shape)
print("Forma de y:", y.shape)

In [ ]:
df_final = pd.DataFrame(X_escalado, columns=features)
df_final['precio_futuro'] = y.values

df_final.to_csv("../data/processed/tsla_processed.csv", index=False)

print("Dataset final guardado correctamente")
print(df_final.shape)
df_final.head()

---
## PARTE 2: ANÁLISIS EXPLORATORIO DE DATOS (EDA)
### Responsable: Persona 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/tsla_processed.csv')
df.head()

In [ ]:
print('Filas y columnas:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)
print('\nValores nulos por columna:')
print(df.isnull().sum())
print('\nRegistros duplicados:', df.duplicated().sum())

In [ ]:
estadisticas = df.describe().T
estadisticas['mediana'] = df.median(numeric_only=True)
estadisticas

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(x=np.arange(len(df)), y=df['precio_futuro'])
plt.title('Evolución del precio futuro de TSLA')
plt.xlabel('Orden cronológico de las observaciones')
plt.ylabel('Precio futuro (USD)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df['precio_futuro'], bins=35, kde=True, color='steelblue')
plt.title('Distribución del precio futuro')
plt.xlabel('Precio futuro (USD)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,3.8))
sns.boxplot(x=df['precio_futuro'], color='steelblue')
plt.title('Diagrama de caja del precio futuro')
plt.xlabel('Precio futuro (USD)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df['retorno_diario'], bins=40, kde=True, color='steelblue')
plt.title('Distribución del retorno diario estandarizado')
plt.xlabel('Retorno diario estandarizado')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,3.8))
sns.boxplot(x=df['volume'],color='steelblue')
plt.title('Diagrama de caja del volumen estandarizado')
plt.xlabel('Volumen estandarizado')
plt.tight_layout()
plt.show()

In [ ]:
correlacion = df.corr(numeric_only=True)

plt.figure(figsize=(13,10))
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 7})
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

correlacion_objetivo = (
    correlacion['precio_futuro']
    .drop('precio_futuro')
    .sort_values(ascending=False)
)
correlacion_objetivo

In [ ]:
plt.figure(figsize=(9,6))
correlacion_objetivo.sort_values().plot(kind='barh')
plt.title('Correlación con el precio futuro')
plt.xlabel('Coeficiente de correlación')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

In [ ]:
variables = ['close', 'precio_anterior', 'sma_5', 'volume']

for variable in variables:
    plt.figure(figsize=(7.5,5))
    sns.scatterplot(data=df, x=variable, y='precio_futuro', alpha=0.45, s=25)
    plt.title(f'{variable} frente a precio_futuro')
    plt.xlabel(f'{variable} (estandarizada)')
    plt.ylabel('Precio futuro (USD)')
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(7.5,5))
sns.scatterplot(data=df, x='volatilidad_5', y='retorno_diario', alpha=0.45, s=25)
plt.title('Volatilidad de 5 días frente a retorno diario')
plt.xlabel('Volatilidad estandarizada')
plt.ylabel('Retorno diario estandarizado')
plt.tight_layout()
plt.show()

### Conclusión del EDA

El dataset está limpio y cumple con el tamaño solicitado. Las variables más relacionadas con el precio futuro son las vinculadas al nivel reciente del precio: apertura, máximo, mínimo, cierre, precio anterior y medias móviles. El retorno, la volatilidad y el volumen aportan una perspectiva diferente sobre los cambios, el riesgo y la actividad del mercado.

---
## PARTE 3: ENTRENAMIENTO DE MODELOS DE REGRESIÓN
### Responsables: Persona 3 y Persona 4

**Modelos a entrenar:**
1. OLS (Linear Regression)
2. Ridge Regression
3. Lasso Regression
4. Bayesian Regression
5. KNN Regression
6. Random Forest Regression
7. SVM Regression
8. Neural Network MLP Regression

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rutas_posibles = [
    '../data/processed/tsla_processed.csv',
    '../../data/processed/tsla_processed.csv',
    'data/processed/tsla_processed.csv',
    'tsla_processed.csv'
]

ruta_encontrada = None
for ruta in rutas_posibles:
    if os.path.exists(ruta):
        ruta_encontrada = ruta
        break

if ruta_encontrada:
    df = pd.read_csv(ruta_encontrada)
    print(f"Dataset cargado correctamente desde: {os.path.abspath(ruta_encontrada)}")
    print(f"Dimensiones del dataset: {df.shape}")
else:
    raise FileNotFoundError("No se encontró 'tsla_processed.csv'. Verifica que el archivo esté en la carpeta data/processed/")

X = df.drop(columns=['precio_futuro'])
y = df['precio_futuro']

split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"\nRegistros de entrenamiento (Train): {len(X_train)}")
print(f"Registros de prueba (Test): {len(X_test)}")
print(f"Proporción: 80% train / 20% test")

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

ols = LinearRegression()
ols.fit(X_train, y_train)

ridge_params = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
grid_ridge = GridSearchCV(Ridge(), ridge_params, cv=tscv, scoring='neg_mean_squared_error')
grid_ridge.fit(X_train, y_train)
best_ridge = grid_ridge.best_estimator_

lasso_params = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
grid_lasso = GridSearchCV(Lasso(max_iter=10000), lasso_params, cv=tscv, scoring='neg_mean_squared_error')
grid_lasso.fit(X_train, y_train)
best_lasso = grid_lasso.best_estimator_

bayes = BayesianRidge()
bayes.fit(X_train, y_train)

knn_params = {
    'n_neighbors': [3, 5, 7, 10, 15],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
grid_knn = GridSearchCV(KNeighborsRegressor(), knn_params, cv=tscv, scoring='neg_mean_squared_error')
grid_knn.fit(X_train, y_train)
best_knn = grid_knn.best_estimator_

print("--- MEJORES HIPERPARÁMETROS ---")
print(f"Ridge: {grid_ridge.best_params_}")
print(f"Lasso: {grid_lasso.best_params_}")
print(f"KNN: {grid_knn.best_params_}")

In [ ]:
configuraciones_rf = [
    {"n_estimators": 50,  "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": None},
]

for config in configuraciones_rf:
    modelo_rf = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42
    )
    modelo_rf.fit(X_train, y_train)
    pred = modelo_rf.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"n_estimators={config['n_estimators']}, max_depth={config['max_depth']} -> "
          f"MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

In [ ]:
configuraciones_svm = [
    {"kernel": "linear", "C": 1},
    {"kernel": "rbf",    "C": 1},
    {"kernel": "rbf",    "C": 10},
]

for config in configuraciones_svm:
    modelo_svm = SVR(kernel=config["kernel"], C=config["C"])
    modelo_svm.fit(X_train, y_train)
    pred = modelo_svm.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"kernel={config['kernel']}, C={config['C']} -> "
          f"MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

In [ ]:
configuraciones_mlp = [
    {"hidden_layer_sizes": (50,),     "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (100, 50), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (100, 50), "learning_rate_init": 0.01},
]

for config in configuraciones_mlp:
    modelo_mlp = MLPRegressor(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        learning_rate_init=config["learning_rate_init"],
        max_iter=2000,
        random_state=42
    )
    modelo_mlp.fit(X_train, y_train)
    pred = modelo_mlp.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"capas={config['hidden_layer_sizes']}, lr={config['learning_rate_init']} -> "
          f"MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

---
## PARTE 4: EVALUACIÓN Y COMPARACIÓN DE MODELOS

In [ ]:
modelos = {
    "OLS (Linear Regression)": ols,
    f"Ridge (alpha={grid_ridge.best_params_['alpha']})": best_ridge,
    f"Lasso (alpha={grid_lasso.best_params_['alpha']})": best_lasso,
    "Bayesian Ridge": bayes,
    f"KNN (k={grid_knn.best_params_['n_neighbors']}, weights='{grid_knn.best_params_['weights']}', p={grid_knn.best_params_['p']})": best_knn,
    "Random Forest (n=100, depth=10)": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42).fit(X_train, y_train),
    "SVM (kernel=linear, C=1)": SVR(kernel="linear", C=1).fit(X_train, y_train),
    "MLP (capas=50, lr=0.001)": MLPRegressor(hidden_layer_sizes=(50,), learning_rate_init=0.001, max_iter=2000, random_state=42).fit(X_train, y_train)
}

resultados = []

for nombre, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    resultados.append({
        "Modelo": nombre,
        "MAE ($)": round(mae, 2),
        "MSE": round(mse, 2),
        "RMSE ($)": round(rmse, 2),
        "R²": round(r2, 4)
    })

df_resultados = pd.DataFrame(resultados).sort_values(by="R²", ascending=False)

print("="*80)
print("TABLA COMPARATIVA DE MODELOS")
print("="*80)
print(df_resultados.to_string(index=False))
print("="*80)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_test.values, label='Precio Real TSLA', color='black', linewidth=2)

for nombre, modelo in modelos.items():
    plt.plot(modelo.predict(X_test), label=nombre, alpha=0.7, linestyle='--')

plt.title('Comparación de Predicciones en el Conjunto de Test')
plt.xlabel('Días del conjunto de prueba')
plt.ylabel('Precio Futuro ($ USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## PARTE 5: PREDICCIÓN CON NUEVOS DATOS

In [ ]:
mejor_modelo = SVR(kernel="linear", C=1)
mejor_modelo.fit(X_train, y_train)

def predecir_precio_futuro(ruta_csv_nuevo, modelo, columnas_esperadas):
    """
    Recibe la ruta a un CSV nuevo (con el mismo formato/columnas que el dataset original,
    ya preprocesado y escalado igual que tsla_processed.csv) y devuelve las predicciones.
    """
    datos_nuevos = pd.read_csv(ruta_csv_nuevo)

    faltantes = set(columnas_esperadas) - set(datos_nuevos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas en el CSV nuevo: {faltantes}")

    X_nuevo = datos_nuevos[columnas_esperadas]
    predicciones = modelo.predict(X_nuevo)
    return predicciones

print("Modelo SVM lineal entrenado y listo para predicciones.")

In [ ]:
X_test.head(5).to_csv("prueba.csv", index=False)

columnas = X_train.columns.tolist()
resultado = predecir_precio_futuro("prueba.csv", mejor_modelo, columnas)
valores_reales = y_test.head(5).values

print("Prueba de predicción con 5 muestras:")
print("-" * 40)
for i in range(len(resultado)):
    print(f"Predicción: ${resultado[i]:.2f} | Real: ${valores_reales[i]:.2f}")

---
## DOCUMENTO DE INFRAESTRUCTURA DEL PROYECTO

### 1. Gráficas del EDA
Las gráficas del EDA se encuentran en la PARTE 2 de este notebook, incluyendo:
- Evolución del precio futuro
- Distribución y diagrama de caja del precio
- Distribución del retorno diario y volumen
- Matriz de correlación
- Gráficas de dispersión (close, precio_anterior, sma_5, volume vs precio_futuro)
- Volatilidad frente a retorno

### 2. Proporción de entrenamiento y prueba
- **80% Entrenamiento:** 1,299 datos
- **20% Prueba:** 325 datos
- Se mantuvo el **orden cronológico** (no aleatorio) ya que se trata de una serie de tiempo.
- Validación cruzada con `TimeSeriesSplit(n_splits=5)`

### 3. Descripción de los algoritmos de regresión

| Modelo | Descripción |
|---|---|
| **OLS** | Mínimos cuadrados ordinarios. Encuentra la recta/plano que minimiza la suma de errores al cuadrado. |
| **Ridge** | Regresión con regularización L2. Reduce magnitud de coeficientes para evitar sobreajuste. |
| **Lasso** | Regresión con regularización L1. Puede eliminar features irrelevantes (coeficientes a 0). |
| **Bayesian Ridge** | Estima parámetros de regularización mediante distribuciones a priori. |
| **KNN** | Predice usando el promedio ponderado de los k vecinos más cercanos en el espacio de features. |
| **Random Forest** | Ensemble de árboles de decisión. Cada árbol aprende reglas y se promedian las predicciones. |
| **SVM** | Busca la función lineal/no lineal que mejor ajusta dentro de un margen de tolerancia. |
| **MLP** | Red neuronal totalmente conectada. Ajusta pesos mediante retropropagación. |

### 4. Hiperparámetros empleados

| Modelo | Hiperparámetros | Valor Óptimo |
|---|---|---|
| Ridge | alpha | 1.0 |
| Lasso | alpha | 1.0 |
| KNN | n_neighbors, weights, p | 5, distance, 1 |
| Random Forest | n_estimators, max_depth | 100, 10 |
| SVM | kernel, C | linear, 1 |
| MLP | hidden_layer_sizes, learning_rate_init | (50,), 0.001 |

### 5. Métricas y estadísticas de rendimiento

| Modelo | MAE ($) | RMSE ($) | R² |
|---|---|---|---|
| SVM (Linear) | 9.44 | 12.07 | 0.9574 |
| Lasso | 9.67 | 12.51 | 0.9543 |
| Bayesian Ridge | 9.92 | 12.92 | 0.9512 |
| OLS | 9.93 | 12.93 | 0.9511 |
| Ridge | 10.02 | 12.95 | 0.9510 |
| Random Forest | 10.11 | 13.12 | 0.9497 |
| MLP | 10.87 | 13.99 | 0.9428 |
| KNN | 16.57 | 21.53 | 0.8645 |

### 6. Funcionamiento del sistema de predicción

El sistema de predicción funciona de la siguiente manera:

1. **Entrenamiento:** El modelo SVM lineal se entrena con datos históricos de TSLA (2020-2026) que incluyen 11 features escaladas.

2. **Predicción:** La función `predecir_precio_futuro()` recibe un CSV con datos nuevos (ya preprocesados y escalados), valida que tenga las columnas correctas, y genera predicciones del precio del día siguiente.

3. **Limitaciones:** Para evaluar con datos reales del día actual, es necesario replicar el pipeline completo de preprocesamiento (medias móviles, retorno, volatilidad) y aplicar el mismo StandardScaler ajustado originalmente.

### 7. Notas técnicas para la exposición

- **StandardScaler** se justifica porque SVM, KNN y MLP son sensibles a la escala de las variables.
- Las **medias móviles** y **volatilidad** son indicadores técnicos comunes en análisis financiero.
- La **variable objetivo** es el precio del día siguiente (horizonte de predicción de 1 día).
- Se usó **TimeSeriesSplit** para validación cruzada, evitando filtrar información del futuro al pasado.